# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
!wget -qO- https://astral.sh/uv/install.sh | sh

import os
# Make uv findable in subsequent cells
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# Verify
!which uv && uv --version

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/usr/local/bin/uv
uv 0.11.8 (x86_64-unknown-linux-gnu)


In [2]:
constraints = """torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57
"""

with open("/content/constraints.txt", "w") as f:
    f.write(constraints)

!cat /content/constraints.txt

torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57


In [3]:
!uv pip install --system torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 100ms


In [4]:
import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("CUDA version torch was built with:", torch.version.cuda)

torch version: 2.5.1+cu121
CUDA available: True
Device count: 1
Device name: NVIDIA A100-SXM4-80GB
CUDA version torch was built with: 12.1


In [5]:
!uv pip install --system \
    sympy numpy transformers vllm tqdm bitsandbytes \
    antlr4-python3-runtime==4.11.1 accelerate \
    -c /content/constraints.txt

Using Python 3.12.13 environment at: /usr
Checked 8 packages in 108ms


In [6]:
import vllm
print("vllm version:", vllm.__version__)

# Re-check torch — make sure the vllm install didn't quietly change it
import torch
print("torch version after vllm install:", torch.__version__)
print("CUDA still available:", torch.cuda.is_available())

vllm version: 0.7.3
torch version after vllm install: 2.5.1+cu121
CUDA still available: True


In [7]:
print("Now go to Runtime → Restart session, then continue with your model loading code.")

Now go to Runtime → Restart session, then continue with your model loading code.


In [8]:
import torch
import vllm
print("Post-restart checks:")
print("  torch:", torch.__version__)
print("  CUDA available:", torch.cuda.is_available())
print("  Device:", torch.cuda.get_device_name(0))
print("  vllm:", vllm.__version__)
print("  GPU memory free:", round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), "GB")

Post-restart checks:
  torch: 2.5.1+cu121
  CUDA available: True
  Device: NVIDIA A100-SXM4-80GB
  vllm: 0.7.3
  GPU memory free: 84.65 GB


### Run the cell below every time to activate the installed environment.

In [9]:
# activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [10]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
MAX_TOKENS  = 32768
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [11]:
DATA_PATH   = "/content/public.jsonl"
OUTPUT_PATH = "/content/starter_results.jsonl"

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [12]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [13]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    trust_remote_code=True,
    max_model_len=16384,
    gpu_memory_utilization=0.5,
    max_num_batched_tokens=32768,
    max_num_seqs=16,            # dropped from 256 — see note below
    enable_prefix_caching=False,
    disable_log_stats=True,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

INFO 05-01 22:03:53 weight_utils.py:270] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 16.557615 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-01 22:03:56 model_runner.py:1115] Loading model weights took 2.4940 GB
INFO 05-01 22:03:59 worker.py:267] Memory profiling takes 2.39 seconds
INFO 05-01 22:03:59 worker.py:267] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.50) = 39.63GiB
INFO 05-01 22:03:59 worker.py:267] model weights take 2.49GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 2.48GiB; the rest of the memory reserved for KV Cache is 34.56GiB.
INFO 05-01 22:03:59 executor_base.py:111] # cuda blocks: 15729, # CPU blocks: 1820
INFO 05-01 22:03:59 executor_base.py:116] Maximum concurrency for 16384 tokens per request: 15.36x
INFO 05-01 22:04:02 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:06<00:00,  1.37s/it]

INFO 05-01 22:04:08 model_runner.py:1562] Graph capturing finished in 7 secs, took 0.22 GiB
INFO 05-01 22:04:08 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 12.42 seconds
Model loaded.


In [15]:
import vllm, transformers, tokenizers
print("vllm:", vllm.__version__)              # 0.7.3 (unchanged)
print("transformers:", transformers.__version__)  # should be 4.48-57
print("tokenizers:", tokenizers.__version__)  # likely auto-downgraded with transformers

import torch
print("torch:", torch.__version__)            # 2.5.1+cu121 (constraints holding)
print("CUDA available:", torch.cuda.is_available())

vllm: 0.7.3
transformers: 4.56.2
tokenizers: 0.22.2
torch: 2.5.1+cu121
CUDA available: True


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [16]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions...


Processed prompts: 100%|██████████| 5/5 [04:33<00:00, 54.69s/it, est. speed input: 3.10 toks/s, output: 97.33 toks/s]


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few even positive whole numbers are. The first one is 2, right? Then 4, 6, 8, and so on. Wait, positive even whole numbers, so they start at 2 and g ...

── Response 1 (id=1) ──
Okay, let's try to figure out this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s^2 + a^2) ds. Hmm, first I need to recall how to compute integrals of this form. 

Wait, the integral of 1/(s^2 + b^2) ds from -infty to +infty is a standard integral. I remember that the integral of 1/(x^2 + c^2) from -infty to infty is (pi)/c. Let me conf ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about the temperature of the turkey cooling 

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [17]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

ModuleNotFoundError: No module named 'judger'

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!